In [1]:
import pandas as pd

In [2]:
# read in the sample for class sessions as a basis for all information for class sessions
class_sessions_df = pd.read_parquet('Sample/sample_classes.parquet')
sample_aps_df = pd.read_parquet('Sample/sample_aps.parquet')
sample_classrooms_df = pd.read_parquet('Sample/sample_classrooms.parquet')


In [ ]:
#print(class_sessions_df.head())
print(class_sessions_df.dtypes)
print(sample_aps_df.dtypes)
print(sample_classrooms_df.dtypes)

In [3]:
#Adding additional columns to the DataFrame
# Ensure string columns use Pandas' dedicated string dtype
class_sessions_df['group'] = pd.Series(dtype="string")

# Set numerical columns with nullable integer dtype (Int64)
class_sessions_df['number_aps'] = pd.Series(dtype="Int64")
class_sessions_df['number_radios'] = pd.Series(dtype="Int64")
class_sessions_df['number_beacons'] = pd.Series(dtype="Int64")
class_sessions_df['max_stations'] = pd.Series(dtype="Int64")
class_sessions_df['min_stations'] = pd.Series(dtype="Int64")
class_sessions_df['average_cu'] = pd.Series(dtype="Int64")
class_sessions_df['median_cu'] = pd.Series(dtype="Int64")

# Compute class_duration in SECONDS (float since it involves time differences)
class_sessions_df['class_duration'] = (class_sessions_df['end_time'] - class_sessions_df['start_time']).dt.total_seconds().astype("Float64")

# Ensure duration_75 and duration_50 are also float types but nullable
class_sessions_df['duration_75'] = pd.Series(dtype="Float64")
class_sessions_df['duration_50'] = pd.Series(dtype="Float64")


In [ ]:

# Display the updated DataFrame
print(class_sessions_df.head())
print(class_sessions_df.dtypes)


In [4]:
# Count occurrences of each Building-Room in sample_aps_df
aps_count = sample_aps_df['Building-Room'].value_counts()

# Map these counts to the number_aps column in class_sessions_df
class_sessions_df['number_aps'] = class_sessions_df['location'].map(aps_count).astype("Int64")


In [ ]:
#Group A, B, and C are from the sample selection based on classroom size
# Create a mapping from sample_classrooms_df: Location -> Group
location_to_group = sample_classrooms_df.set_index('Location')['Group']

# Map the Group values to class_sessions_df based on location
class_sessions_df['group'] = class_sessions_df['location'].map(location_to_group).astype("string")


In [7]:
print(class_sessions_df.head())

    location    course                start_time                  end_time  \
0  BNSN-W003  NURS 296 2024-11-04 08:30:00-07:00 2024-11-04 09:50:00-07:00   
1  BNSN-W003  CHEM 105 2024-11-05 08:00:00-07:00 2024-11-05 08:50:00-07:00   
2  BNSN-W003  CHEM 105 2024-11-05 09:00:00-07:00 2024-11-05 09:50:00-07:00   
3  BNSN-W003  CHEM 105 2024-11-05 10:00:00-07:00 2024-11-05 10:50:00-07:00   
4  BNSN-W003  CHEM 106 2024-11-05 12:00:00-07:00 2024-11-05 12:50:00-07:00   

   enrolled  capacity group  number_aps  number_radios  number_beacons  \
0         8        25     A           1           <NA>            <NA>   
1        21        25     A           1           <NA>            <NA>   
2        22        25     A           1           <NA>            <NA>   
3        21        25     A           1           <NA>            <NA>   
4        14        25     A           1           <NA>            <NA>   

   max_stations  min_stations  average_cu  median_cu  class_duration  \
0          <NA

In [ ]:
# populate the min/max scount from all_class_beacons_processed.parquet
# need to identify the APs that do not have any captures for the class and remove it from the sample